In [ ]:
import hashlib
from merkle_tree import MerkleTree
from rescue import RescuePrime

In [ ]:
# 1. Define a clean, raw byte SHA-256 helper
def sha256_bytes(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()



# Test dataset with an odd number of transactions to validate edge-case duplication
transactions = [b"Tx0_Alpha", b"Tx1_Beta", b"Tx2_Gamma", b"Tx3_Delta", b"Tx4_Epsilon"]

# 1. Initialize and build the tree
tree = MerkleTree(transactions, hash=sha256_bytes)
print(f"Merkle Root Generated (Hex): {tree.root.hex()}")

# 2. Extract a proof for "Tx2_Gamma" (Index 2)
target_idx = 2
target_leaf = transactions[target_idx]
proof = tree.get_proof(target_idx)

print(f"\nGenerated Proof Path for Index {target_idx}:")
for step, (h, side) in enumerate(proof):
    side_str = "Left" if side else "Right"
    print(f"  Layer {step} Sibling: {h.hex()[:16]}... (Position: {side_str})")
    
# 3. Verify the proof
is_valid = tree.verify_proof(target_leaf, proof, tree.root)
print(f"\nIs the proof cryptographically valid? -> {is_valid}")

# 4. Attempt to verify tampered data using the same proof
fake_leaf = b"Tx2_Tampered_Data"
is_fake_valid = tree.verify_proof(fake_leaf, proof, tree.root)
print(f"Did the verification successfully catch the tampered data? -> {not is_fake_valid}")

Merkle Root Generated (Hex): 798f14ed03615400e00418db5bdfc592243bb82865a095a3fa63bc71baf6e2df

Generated Proof Path for Index 2:
  Layer 0 Sibling: 9f5211ab1e9b0b13... (Position: Right)
  Layer 1 Sibling: 373b955412875523... (Position: Left)
  Layer 2 Sibling: b214a69f88558bca... (Position: Right)

Is the proof cryptographically valid? -> True
Did the verification successfully catch the tampered data? -> True


In [ ]:
import random
import time

p = 2**61 - 1
m = 3
c = 1
s = 128
output_len = 1

cpu_rp = RescuePrime(p, m , c, s, enable_gpu=False)
gpu_rp = RescuePrime(p, m, c, s, enable_gpu=True)

print("Generating Batched Merkle Tree Test")
batch_sz = 8192
elements_per_msg = 8
test_batch = [random.randint(0, 255) for _ in range(batch_sz)]

start_cpu = time.perf_counter()
cpu_tree = MerkleTree(test_batch, hash=cpu_rp.hash_batch, is_batched=True)
cpu_result = cpu_tree.root.hex()
cpu_time = time.perf_counter() - start_cpu
print ("CPU Batch Completed in: {cpu_time} seconds with hash {cpu_result}")

start_gpu = time.perf_counter()
gpu_tree = MerkleTree(test_batch, hash=gpu_rp.hash_batch, is_batched=True)
gpu_result = gpu_tree.root.hex()
gpu_time = time.perf_counter() - start_cpu
print ("GPU Batch Completed in: {cpu_time} seconds with hash {cpu_result}")
gpu_tree = MerkleTree (test_batch, hash=gpu_rp.hash_batch, is_batched=True)

if cpu_result == gpu_result:
    print("[PASS] Cryptographic Equivalence Maintained across all 10,000 hashes!")
    speedup = cpu_time / gpu_time
    print(f"[Speedup Factor]: GPU is {speedup:.2f}x FASTER than CPU.")
